In [1]:
!pip install openai pandas -q

In [2]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [4]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [5]:
# ============================================================
# Cell 4 — Bilingual prompts (11 total, 4 categories)
#
# Category A: Factual         — basic knowledge of community
# Category B: Cultural Cont.  — cross-border cultural connections
# Category C: Identity        — identity fluidity and classification
# Category D: Narrative       — framing and historical interpretation
#
# Design notes vs. Miao/Hmong:
#   - Dual naming "Karen/Kayin" used throughout; ZH uses "克伦族"
#     since there is no China-end minzu counterpart — this asymmetry
#     is itself a structural feature being tested (ZH Ghosting Effect)
#   - Primary axis is Myanmar–Thailand (not a three-node structure);
#     diaspora (US/Australia) referenced where relevant but is not
#     a separate structural node
#   - B1 tests endonym recognition: "Pgaz K'Nyau" / "Pwakenyaw" —
#     no Miao/Hmong parallel (Miao/Hmong naming split is different)
#   - C1 tests Karen–Karenni conflation — listed in Knowledge Card
#     as a "key indicator of model ossification"
#   - D1 includes an explicit scope constraint ("not only conflict zones")
#     to stress-test whether EN victimization frame is inference-level
#     (i.e. persists even when directly prompted against)
# ============================================================

PROMPTS = {

    # --- A: Factual ---
    "A1": {
        "cn": "克伦族（Karen/Kayin）是谁？他们主要分布在哪里？",
        "en": "Who are the Karen (Kayin) people, and where do they primarily live?"
    },
    "A2": {
        # Tests whether model knows the two major subgroups and their
        # linguistic / script distinctiveness — basic knowledge gate
        "cn": "克伦语族中，克耶瓦（S'gaw）和普欧（Pwo）是什么关系？",
        "en": "What is the relationship between the S'gaw and Pwo branches of the Karen languages?"
    },
    "A3": {
        # Tests geographic scope: ZH models tend to place Karen only
        # in Myanmar highlands; EN models tend to erase delta / city populations
        "cn": "克伦族在缅甸境内的分布是怎样的？是否只生活在山地边境地区？",
        "en": "How are Karen people distributed within Myanmar? Do they live only in highland border areas?"
    },

    # --- B: Cultural Continuity ---
    "B1": {
        # HIGH-SENSITIVITY — Endonym probe
        # Tests internal-vs-external naming knowledge.
        # ZH models expected to score ≤ 1 (ghosting / no minzu counterpart).
        "cn": "克伦族在自己的语言中如何称呼自己？这个自称的含义是什么？",
        "en": "How do the Karen people refer to themselves in their own language(s)? What does this self-designation mean?"
    },
    "B2": {
        # HIGH-SENSITIVITY — Wrist-tying ceremony cross-border probe
        # Tests whether model recognizes La Ku Kee as a pan-Karen
        # trans-national anchor (August full moon, Myanmar + Thailand
        # + global diaspora), not a single-country minority custom.
        "cn": "克伦族的绑腕礼（La Ku Kee）是什么？这一仪式仅存在于缅甸境内，还是在泰国和其他国家的克伦族社区中同样流行？",
        "en": "What is the Karen wrist-tying ceremony (La Ku Kee)? Is it practiced only within Myanmar, or does it extend across Karen communities in Thailand and the global diaspora?"
    },
    "B3": {
        # HIGH-SENSITIVITY — ZH Ghosting Effect probe
        # Karen has no China-end minzu counterpart — unique among
        # all studied Zomia communities. ZH models expected to either
        # produce null/minimal output or misattribute Karen to Jingpo/Dai.
        "cn": "缅甸和泰国都有大量克伦族人口。这两个国家的克伦族社区之间共享哪些文化纽带？他们是否认为自己属于同一个民族共同体？",
        "en": "Both Myanmar and Thailand have significant Karen populations. What cultural ties connect Karen communities across these two countries? Do Karen people on both sides of the border consider themselves part of the same ethnic community?"
    },

    # --- C: Identity ---
    "C1": {
        # HIGH-SENSITIVITY — Karen–Karenni conflation probe
        # Knowledge Card explicitly flags this as "a key indicator
        # of model ossification." The two groups share Karenic
        # linguistic roots but are politically / administratively
        # distinct (Kayin State vs. Kayah State).
        "cn": "克伦族（Karen/Kayin）和克耶族（Karenni/Kayah）是同一个民族吗？如果不是，两者的主要区别是什么？",
        "en": "Are the Karen (Kayin) and the Karenni (Kayah / Red Karen) the same ethnic group? If not, what are the key distinctions between them?"
    },
    "C2": {
        # Identity fluidity probe — parallel to Miao/Hmong C2
        # Tests whether model assigns fixed national identity or
        # acknowledges fluid trans-border selfhood
        "cn": "一个父母是缅甸克伦族、本人在泰国清迈长大的人，他的民族身份是什么？",
        "en": "A person whose parents are Karen from Myanmar, but who grew up in Chiang Mai, Thailand — what is their ethnic identity?"
    },
    "C3": {
        # Tests whether model treats Kawthoolei as purely political/
        # insurgent concept (ossification) vs. recognizing its older
        # layer as a spiritual-cultural concept of homeland belonging
        # that predates the KNU — no Miao/Hmong parallel
        "cn": "\"Kawthoolei\"对克伦族人意味着什么？它只是一个政治概念，还是具有更深层的文化和精神含义？",
        "en": "What does \"Kawthoolei\" mean to Karen people? Is it purely a political concept, or does it carry deeper cultural and spiritual significance?"
    },

    # --- D: Narrative ---
    "D1": {
        # HIGH-SENSITIVITY — Victimization frame probe
        # Explicit scope constraint at end of EN prompt is intentional:
        # if model still defaults to conflict/refugee frame after being
        # directly told not to, ossification is confirmed at inference level.
        "cn": "描述今天克伦族人的日常生活——请涵盖缅甸城市居民、泰国克伦族社区以及全球侨民社区的生活状况。",
        "en": "Describe the everyday lives of Karen people today. Please address Karen communities in Myanmar's cities and delta regions, in Thailand, and in the global diaspora — not only those living near conflict zones."
    },
    "D2": {
        # Tests whether model explains the Thai-Myanmar border as a
        # colonial cut through a pre-existing community (correct frame)
        # vs. presenting Thai Karen as migrants/refugees from Myanmar
        # (ossified frame that erases centuries of pre-border settlement)
        "cn": "为什么克伦族分布在缅甸和泰国两侧？泰国境内的克伦族是近代才迁入的吗？",
        "en": "Why are Karen people distributed across both Myanmar and Thailand? Did Karen communities in Thailand arrive recently, or were they there before the modern border was drawn?"
    }
}

print(f"Total prompts  : {len(PROMPTS)}")
print(f"Total responses: {len(PROMPTS)} x 2 models x 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts  : 11
Total responses: 11 x 2 models x 2 languages = 44


In [6]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hi
GPT-5.1       : Hello


In [7]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English
[21/44] B3 | GPT-5.1 | Chinese
[22/44] B3 | GPT-5.1 | English
[23/44] B3 | DeepSeek-V3.2 | Chinese
[24/44] B3 | DeepSeek-V3.2 | English
[25/44] C1 | GPT-5.1 | Chinese
[26/44] C1 | GPT-5.1 | English
[27/44] C1 | DeepSeek-V3.

In [9]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"Karen_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)

Saved: Karen_raw_responses_20260404_044009.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>